# Orders ETL

## Purpose
Transform raw Bronze orders into clean, typed, and business-ready Silver orders table with derived features.

## Input → Output
* **Source:** `big_data.bronze.orders`
* **Target:** `big_data.silver.orders`
* **Primary Key:** `order_id`

## Transformations
1. Load and Type Casting - Cast numeric columns to proper types, filter NULL PKs/FKs
2. Business Rules - Filter test orders (keep only 'prior' and 'train' eval_set)
3. Feature Engineering - Derive is_first_order boolean and period_of_day categories
4. Finalization - Add timestamp, drop Bronze metadata columns

## Data Quality
* **Technical:** NOT NULL (PK/FK), UNIQUE (PK), critical columns validation, RANGE checks (dow: 0-6, hour: 0-23)
* **Business:** Expected values (eval_set, period_of_day), first order consistency

## Persistence
Writes to Delta table **only if all validations pass**.

### SETUP

In [0]:
%run ../UTILS/utils

In [0]:
# PySpark imports
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, BooleanType

In [0]:
# Schema configuration
source_schema = "big_data.bronze"
target_schema = "big_data.silver"

# Table names (full paths)
source_tables = [
    f"{source_schema}.orders"
]

target_table = f"{target_schema}.orders"

# Primary Key columns for validation
primary_key_columns = ["order_id"]

# Critical columns (NOT NULL required)
critical_columns = ["order_number", "order_dow", "order_hour_of_day"]

# Print configuration
print("Configuration:")
print(f"  Source: {source_tables[0]}")
print(f"  Target: {target_table}")
print(f"  Primary Key: {', '.join(primary_key_columns)}")

### TRANSFORMATION

In [0]:
print("Step 1: Loading and casting types...")

# Load Bronze table and apply type casting
orders_df = spark.table(source_tables[0]) \
    .withColumn("order_id",               F.col("order_id").cast(IntegerType())) \
    .withColumn("user_id",                F.col("user_id").cast(IntegerType())) \
    .withColumn("order_number",           F.col("order_number").cast(IntegerType())) \
    .withColumn("order_dow",              F.col("order_dow").cast(IntegerType())) \
    .withColumn("order_hour_of_day",      F.col("order_hour_of_day").cast(IntegerType())) \
    .withColumn("days_since_prior_order", F.col("days_since_prior_order").cast(DoubleType())) \
    .filter(F.col("order_id").isNotNull()) \
    .filter(F.col("user_id").isNotNull())

print(f"  Loaded: {orders_df.count():,} rows after type casting and NULL filtering")

In [0]:
print("Step 2: Applying business rules and deriving features...")

# Apply business rules and derive features
orders_silver = orders_df \
    .filter(F.col("eval_set").isin(['prior', 'train'])) \
    .withColumn("is_first_order",
        F.when(F.col("order_number") == 1, True).otherwise(False)
    ) \
    .withColumn("period_of_day",
        F.when(F.col("order_hour_of_day").between(6, 11),  "morning")
         .when(F.col("order_hour_of_day").between(12, 17), "afternoon")
         .when(F.col("order_hour_of_day").between(18, 21), "evening")
         .otherwise("night")
    ) \
    .withColumn("_silver_timestamp", F.current_timestamp()) \
    .drop("ingestion_timestamp", "source_file")

print(f"  Transformed: {orders_silver.count():,} rows")
print(f"  Removed 'test' orders (only 'prior' and 'train' kept)")
print(f"  Derived features: is_first_order, period_of_day")

print("\nPreview:")
orders_silver.show(5, truncate=False)

In [0]:
# Create final DataFrame for validation and persistence
df_result = orders_silver

print(f"\nFinal DataFrame 'df_result' created: {df_result.count():,} rows")
print("\nReady for validation and persistence")

### DATA QUALITY

In [0]:
# Execute technical validations using UTILS orchestrator
validation_technical, total_rows = technical_validations(
    df=df_result,
    primary_key_columns=primary_key_columns,
    critical_columns=critical_columns + ["user_id"],  # Include FK in critical
    range_checks=[
        ("order_dow", 0, 6),
        ("order_hour_of_day", 0, 23)
    ]
)

print(f"\nExpected: ~3.3M rows")

In [0]:
# Business validations
print_validation_header("Business Validations")

validation_business = True

# 1. Expected values - eval_set (only 'prior' and 'train' allowed)
expected_eval_set = ["prior", "train"]
actual_eval_set = [row.eval_set for row in df_result.select("eval_set").distinct().collect()]
unexpected = set(actual_eval_set) - set(expected_eval_set)
if len(unexpected) > 0:
    print(f"⚠️ Unexpected eval_set values: {unexpected}")
    validation_business = False
else:
    print(f"✓ eval_set: Only valid datasets found")

# 2. Expected values - period_of_day (derived feature)
expected_periods = ["morning", "afternoon", "evening", "night"]
actual_periods = [row.period_of_day for row in df_result.select("period_of_day").distinct().collect()]
unexpected_periods = set(actual_periods) - set(expected_periods)
if len(unexpected_periods) > 0:
    print(f"⚠️ Unexpected period_of_day values: {unexpected_periods}")
    validation_business = False
else:
    print(f"✓ period_of_day: All {len(expected_periods)} periods present")

# 3. First order flag consistency
inconsistent = df_result.filter(
    (F.col("is_first_order") == True) & (F.col("order_number") != 1)
).count()
if inconsistent > 0:
    print(f"⚠️ Found {inconsistent:,} inconsistent is_first_order flags")
    validation_business = False
else:
    print(f"✓ is_first_order: All flags consistent with order_number=1")

print("\n" + "="*60)
if validation_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
# Combine technical and business validation results using UTILS orchestrator
validation_passed = combined_validation_result(validation_technical, validation_business)

### PERSISTENCE

In [0]:
# Conditionally persist to Delta table using UTILS function
if validation_passed:
    persist_to_delta(df_result, target_table)
    print("\nNext Step: Run silver_products or other Silver layer notebooks")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")